In [0]:
spark.sql("USE CATALOG 03_gold")
spark.sql("USE SCHEMA default")
from pyspark.sql.functions import (
    col, initcap, regexp_replace, to_date, coalesce, 
    current_timestamp, expr, trim,year,month,dayofmonth,date_format,sum
)
import re

**Customer diamension table**

In [0]:
df = spark.read.table("02_silver.default.customers")

df_dim_customer = df.select(
    "customer_id",
    "customer_name",
    "gender",
    "phone_number"
).dropDuplicates(["customer_id"])

spark.sql("DROP TABLE IF EXISTS 03_gold.default.dim_customer")
df_dim_customer.write.mode("overwrite").saveAsTable("03_gold.default.dim_customer")

**Products diamention table**

In [0]:
df = spark.read.table("02_silver.default.products")

df_dim_product = df.select(
    "product_id",
    "product_name",
    "category",
    "cost_price",
    "marked_price"
).dropDuplicates(["product_id"])

spark.sql("DROP TABLE IF EXISTS 03_gold.default.dim_product")
df_dim_product.write.mode("overwrite").saveAsTable("03_gold.default.dim_product")

**Stores diamension table**

In [0]:
df = spark.read.table("02_silver.default.stores")

df_dim_store = df.select(
    "store_id",
    "store_name",
    "city"
).dropDuplicates(["store_id"])

spark.sql("DROP TABLE IF EXISTS 03_gold.default.dim_store")
df_dim_store.write.mode("overwrite").saveAsTable("03_gold.default.dim_store")

**Dates diamension table**

In [0]:
df = spark.read.table("02_silver.default.sales")

df_dates = df.select(col("transaction_date").alias("date")).dropDuplicates()

df_dim_date = df_dates.select(
    "date",
    year("date").alias("year"),
    month("date").alias("month"),
    dayofmonth("date").alias("day"),
    date_format("date", "E").alias("weekday")
)

spark.sql("DROP TABLE IF EXISTS 03_gold.default.dim_date")
df_dim_date.write.mode("overwrite").saveAsTable("03_gold.default.dim_date")

**Sales fact table**

In [0]:
df_sales = spark.read.table("02_silver.default.sales")
df_products = spark.read.table("02_silver.default.products")

df_fact_sales = df_sales.join(
    df_products,
    "product_id",
    "left"
).select(
    "transaction_id",
    "customer_id",
    "store_id",
    "product_id",
    col("transaction_date").alias("date"),

    "quantity",
    "selling_price",

    (col("quantity") * col("selling_price")).alias("total_sales"),
    (col("selling_price") - col("cost_price")).alias("profit_per_unit"),
    (col("quantity") * (col("selling_price") - col("cost_price"))).alias("total_profit"),
    (col("marked_price") - col("selling_price")).alias("discount")
)

spark.sql("DROP TABLE IF EXISTS 03_gold.default.fact_sales")

df_fact_sales.write.format("delta").mode("overwrite").saveAsTable("03_gold.default.fact_sales")

**Returns fact table**

In [0]:
df_returns = spark.read.table("02_silver.default.returns")

df_fact_returns = df_returns.select(
    "return_id",
    "transaction_id",
    col("return_date").alias("date"),
    "return_reason"
)

spark.sql("DROP TABLE IF EXISTS 03_gold.default.fact_returns")

df_fact_returns.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("03_gold.default.fact_returns")

**Inventory fact table**

In [0]:
df_inventory = spark.read.table("02_silver.default.inventory")

df_fact_inventory = df_inventory.select(
    "product_id",
    "store_id",
    "stock_qty"
)

spark.sql("DROP TABLE IF EXISTS 03_gold.default.fact_inventory")

df_fact_inventory.write.format("delta").mode("overwrite").saveAsTable("03_gold.default.fact_inventory")